# 06 KPI Export\nPurpose: Build KPI tables and Tableau-ready extracts for executive and operational dashboards.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport numpy as np\n\nROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nCLEAN_PATH = ROOT / 'data' / 'processed' / 'loan_clean.csv'\nSCORED_PATH = ROOT / 'data' / 'processed' / '05_default_scored.csv'\nKPI_OUT = ROOT / 'data' / 'processed' / 'kpi_master.csv'\nSEGMENT_OUT = ROOT / 'data' / 'processed' / 'kpi_segment_table.csv'\nTABLEAU_OUT = ROOT / 'data' / 'processed' / 'tableau_input.csv'

In [ ]:
df = pd.read_csv(CLEAN_PATH)\ntotal_loans = len(df)\ndefault_rate = df['Default'].mean() * 100\navg_loan = df['LoanAmount'].mean()\navg_income = df['Income'].mean()\navg_dti = df['DTIRatio'].mean()

In [ ]:
kpi = pd.DataFrame([\n    {'kpi_name': 'Total Loans', 'value': total_loans},\n    {'kpi_name': 'Default Rate %', 'value': round(default_rate, 3)},\n    {'kpi_name': 'Average Loan Amount', 'value': round(avg_loan, 2)},\n    {'kpi_name': 'Average Income', 'value': round(avg_income, 2)},\n    {'kpi_name': 'Average DTI Ratio', 'value': round(avg_dti, 4)},\n])\nkpi.to_csv(KPI_OUT, index=False)\nkpi

In [ ]:
segment = df.groupby(['EmploymentType', 'LoanPurpose'], as_index=False).agg(\n    loans=('LoanID', 'count'),\n    defaults=('Default', 'sum'),\n    default_rate=('Default', 'mean'),\n    avg_loan=('LoanAmount', 'mean'),\n)\nsegment['default_rate'] = (segment['default_rate'] * 100).round(3)\nsegment = segment.sort_values('default_rate', ascending=False)\nsegment.to_csv(SEGMENT_OUT, index=False)\nsegment.head(20)

In [ ]:
tableau_df = df.copy()\nif SCORED_PATH.exists():\n    scored = pd.read_csv(SCORED_PATH)\n    if {'LoanID', 'default_probability'} <= set(scored.columns):\n        tableau_df = tableau_df.merge(\n            scored[['LoanID', 'default_probability', 'risk_band']],\n            on='LoanID',\n            how='left',\n        )\ntableau_df.to_csv(TABLEAU_OUT, index=False)\nprint('saved:', KPI_OUT)\nprint('saved:', SEGMENT_OUT)\nprint('saved:', TABLEAU_OUT)